# Camera Manual RAG System

## Introduction

Large Language Models (LLMs) excel in general language tasks but suffer from a lack of domain precision, e.g., hallucinating facts and not being contextually grounded in specialized domains like medicine or law. Retrieval-Augmented Generation (RAG) is a solution to these issues.

In this project, a RAG system was developed and evaluated using a niche yet practically valuable domain: ***digital camera user manuals***. Specifically, manuals for four Olympus camera models were collected and processed to build a custom knowledge base. The goal is to explore how well a RAG system can answer technical questions based on these documents, compared to a standalone LLM.




# 0. Prepration Before Start


## 0.1 Install Dependencies


In [ ]:
!pip install -q langchain-community pypdf sentence-transformers ctransformers chromadb nltk pi_heif \
    unstructured[local-inference] unstructured_inference google-generativeai cohere rank_bm25 pdf2image pytesseract
!apt-get install -y poppler-utils tesseract-ocr
!pip install ragas langchain-core langchain

## 0.2 Imports and Setup

In [ ]:
import os
import shutil
import io
import nltk
import cohere
import pytesseract
import google.generativeai as genai
import getpass
from PIL import Image
from nltk.tokenize import sent_tokenize
from pdf2image import convert_from_path
from google.colab import files
from langchain.schema import Document
from unstructured.partition.pdf import partition_pdf
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.retrievers import BM25Retriever, EnsembleRetriever

nltk.download('punkt')

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Enter your Gemini API key: ")
genai.configure(api_key=GEMINI_API_KEY)

## 0.3 Upload PDFs

Digital camera operation manuals presents a compelling test case for RAG. They are rich in procedural knowledge and domain-specific terminology, often combining textual instructions with extensive visual elements, such as labeled images and symbolic icons. This semi-structured, image-heavy documentation poses challenges to standard LLMs, which lack access to proprietary documentation and are not optimized for interpreting symbolic or visual semantics.

Our project uses four official Olympus camera manuals, sourced from their website. These manuals contain detailed operational guidance, settings explanations, and extensive iconography. While LLMs may generalize across similar consumer electronics topics, they typically lack the specific, up-to-date technical content found in proprietary documents such as these.

In [ ]:
uploaded_files = files.upload()

# 1. Data Preprocessing:

## 1.1 Chunking

The next step in the RAG pipeline is the extraction and preprocessing of document content for embedding and retrieval.

First, the *partition_pdf( ) from the unstructured Python package* split the document into images, tables, and paragraphs. Each component contains metadata that includes the camera model name, helping source the information from which camera model, page and type (e.g., "Text", "Image", "Table").

Second, the components are processed according to their type. For images, optical character recognition (OCR) using the *pytesseract library* is employed to extract textual content embedded in images. This is because the camera manual uses labelled diagrams and symbolic icons to present operational instructions. Text components, including extracts from OCR and text-based components, are enveloped in Document object, and tagged with the model and page information.

Third, the Document objects undergo semantic chunking via a custom semantic_chunking() function. Instead of using fixed-sized tokens or character length, this function chunks content by gathering successive sentences up until the highest word limit is reached. Sentence boundaries are recognized via the *sent_tokenize()* function from the *NLTK library*, which provides accurate segmentation by consideration of punctuation and grammatical structure. This chunking scheme preserves the semantic consistency of each chunk, improving the relevance of retrieved passages in later stages of the RAG pipeline. Each chunk is also assigned a unique identifier for indexing and traceability.

In [ ]:
# Initialize a list to store all chunks from all documents
all_chunks = []
os.makedirs("/content/image_chunks", exist_ok=True)

# Function to split documents into semantic chunks by sentence groups (based on max word count)
def semantic_chunking(documents, max_words=100):
    chunks = []
    for doc in documents:
        # Split the document content into sentences
        sentences = sent_tokenize(doc.page_content)
        buffer, count = [], 0

        # Group sentences into chunks based on word count
        for sent in sentences:
            buffer.append(sent)
            count += len(sent.split())
            if count >= max_words:
                chunks.append(Document(" ".join(buffer), metadata=doc.metadata))
                buffer, count = [], 0
        # Handle any remaining sentences
        if buffer:
            chunks.append(Document(" ".join(buffer), metadata=doc.metadata))
    return chunks

# Process each uploaded PDF file
for filename in uploaded_files:
    path = f"/content/{filename}"
    model_name = os.path.splitext(filename)[0]
    elements = partition_pdf(
        filename=path,
        strategy="hi_res",
        extract_images_in_pdf=True,
        infer_table_structure=True
    )

    documents = []
    for elem in elements:
        category = elem.category or "Unknown"
        text = (elem.text or "").strip()
        page = getattr(elem.metadata, "page_number", 0)


        # Prepare metadata for each extracted document element
        metadata = {"model": model_name, "page": page, "type": category}


        # If the element is an image and has an extractable image object
        if category == "Image" and hasattr(elem, "image"):
            filename_img = f"{model_name}_page{page}_img.png"
            filepath = f"/content/image_chunks/{filename_img}"
            elem.image.save(filepath, format="PNG")
            image = Image.open(filepath)
            ocr_text = pytesseract.image_to_string(image)

            # Apply OCR to the image
            if ocr_text.strip():
                metadata["source"] = "OCR"
                metadata["image_path"] = filepath
                documents.append(Document(
                    page_content=f"[MODEL: {model_name}] [PAGE: {page}] [SOURCE: OCR]\nImage OCR: {ocr_text.strip()}",
                    metadata=metadata))

        # If the element has extractable text content
        if text:
            documents.append(Document(
                page_content=f"[MODEL: {model_name}] [PAGE: {page}]\n{text}",
                metadata=metadata))

    # Chunk all text-based (and OCR-based) documents
    chunks = semantic_chunking(documents)
    print(f"{filename} extracted {len(chunks)} chunks")

    for i, chunk in enumerate(chunks):
      chunk.metadata["id"] = f"{model_name}_{i}"

    # Add the chunks to the global all_chunks list
    all_chunks.extend(chunks)

After all chunks have been processed, they are saved to Google Drive. This is necessary because the runtime environment is hosted on Google Colab, and once the session is disconnected, reprocessing all the chunks would take approximately 4 to 5 hours.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
import pickle

In [ ]:
# Step 2: Set common project directory
project_name = "rag_project"
base_dir = f"/content/drive/MyDrive/{project_name}"
os.makedirs(base_dir, exist_ok=True)

# Step 3: Save all_chunks.pkl
chunks_path = os.path.join(base_dir, "all_chunks.pkl")
with open(chunks_path, "wb") as f:
    pickle.dump(all_chunks, f)
print(f"all_chunks saved to: {chunks_path}")

## 1.2 Embedding And Vector Database Storage

The processed all_chuncks are embedded in this stage and stored in Google Drive for future use.

For the embedding step, the model of HuggingFaceEmbeddings - [*intfloat/e5-base-v2*](https://huggingface.co/intfloat/e5-base-v2)  is used, which is a kind of **state-of-the-art embedding model** in the field of dense retrieval and is optimized for semantic search tasks. It encoded both queries and documents into the same vector space, making semantic similarity more accurate when the phrasing differs. Also, unlike traditional keyword-based search, dense retrieval enables the system to retrieve semantically relevant information even when the wording differs, making it effective. Other embedding models were tested, such as BAAI/bge-base-en-v1.5. However, when trying in this project, the performance of intfloat/e5-base-v2 was found better.

After embedding, the vectors are stored in *Chorma*, an open-sourced vector database. It supports efficient similarity search using cosine distance and allows each document to be tagged with rich metadata, such as the source model and page number. It is also natively compatible with both LangChain and Hugging Face libraries, which significantly improves development efficiency and integration.

Furthermore, the use of a persist_directory enables the vector store to be saved and reloaded directly from cloud storage.


In [ ]:
# Define a temporary persist_dir in the local Colab environment
temp_persist_dir = "/content/temp_olympus_vector_db"

# Safely delete the temporary directory if it exists from a previous run
if os.path.exists(temp_persist_dir):
    shutil.rmtree(temp_persist_dir)
    print("Temporary persist_dir deleted. Will rebuild.")

# Create the temporary directory
os.makedirs(temp_persist_dir, exist_ok=True)

embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/e5-base-v2",       # A pre-trained semantic embedding model from Hugging Face, optimized for dense retrieval tasks
    encode_kwargs={"normalize_embeddings": True} # Ensures all output vectors are L2-normalized (unit length), improving consistency and performance in cosine similarity-based retrieval
)

# Split documents into smaller batches (maximum 5000 per batch)
BATCH_SIZE = 5000
batches = [all_chunks[i:i + BATCH_SIZE] for i in range(0, len(all_chunks), BATCH_SIZE)]

# Initialize the Chroma vector store
vectorstore = None
total_batches = len(batches)

for i, batch in enumerate(batches):
    print(f"Processing batch {i+1}/{total_batches}, total {len(batch)} documents...")

    if vectorstore is None:
        # Create the vector store with the first batch in the temporary directory
        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embedding_model,
            persist_directory=temp_persist_dir,
            collection_name="olympus_manual"
        )
    else:
        # Add subsequent batches to the vector store in the temporary directory
        vectorstore.add_documents(batch)

# Save the vector store to disk in the temporary directory
vectorstore.persist()

print("All documents processed and saved successfully to temporary directory.")

# Define the final persist_dir on Google Drive
base_dir = f"/content/drive/MyDrive/{project_name}"
final_persist_dir = os.path.join(base_dir, "olympus_vector_db")

# Safely delete the existing directory on Google Drive before copying
if os.path.exists(final_persist_dir):
    shutil.rmtree(final_persist_dir)
    print("Existing Google Drive persist_dir deleted.")

# Copy the successfully created vector store from the temporary directory to Google Drive
shutil.copytree(temp_persist_dir, final_persist_dir)

print(f"Vector store successfully copied to Google Drive: {final_persist_dir}")

# Clean up the temporary directory (optional, but good practice)
shutil.rmtree(temp_persist_dir)
print(f"Temporary directory {temp_persist_dir} removed.")

If the previous cell has been executed and the vectors have been successfully saved to Google Drive, the above cells can be skipped and run this section directly after reconnecting.

In [ ]:
def load_everything(project_name="rag_project"):

    # Define paths
    base_dir = f"/content/drive/MyDrive/{project_name}"
    chunks_path = os.path.join(base_dir, "all_chunks.pkl")
    persist_dir = os.path.join(base_dir, "olympus_vector_db")

    # Load all_chunks from pickle
    with open(chunks_path, "rb") as f:
        all_chunks = pickle.load(f)

    # Recreate embedding model
    from langchain_community.embeddings import HuggingFaceEmbeddings
    embedding_model = HuggingFaceEmbeddings(
        model_name="intfloat/e5-base-v2",
        encode_kwargs={"normalize_embeddings": True}
    )

    # Reload persisted vectorstore
    from langchain_community.vectorstores import Chroma
    vectorstore = Chroma(
        persist_directory=persist_dir,
        embedding_function=embedding_model,
        collection_name="olympus_manual"
    )

    print(f"Loaded all_chunks from: {chunks_path}")
    print(f"Loaded vectorstore from: {persist_dir}")
    return all_chunks, vectorstore, embedding_model

all_chunks, vectorstore, embedding_model = load_everything()

# 2. Retrieval & Reranking

## 2.1 Query Processing & Utility Functions

Moving to the retrieval steps, a few things are done to refine the user's query and ensure the RAG system interacts effectively with the camera manuals.

1.   normalize_query(query)

    The model_aliases are defined to handle the variance in how users might call specific camera models. The queries are then normalizd by converting them to lowercase and replacing the model name with a standard name. This ensures the system retrieves relevant information regardless of minor differences in the user's input (case, variant spellings of the model names).
2.   rewrite_query(user_query, llm)

    This function used LLM to rewrite the user's input into a form that would sound as if it is to be asked as part of a camera manual. This transformation renders the user's common language into a more technical language of the documentation.
3.   extract_models(query)

    This function extracts the specific camera models mentioned in the user's query. It helps to retrieve the correct answer from the correct camera model as there are four manuals in this project.
4.   is_comparison_query(query) & is_general_concept_query(query)

    These two functions classify user's quries, enabling it to tailor etrieval and generation strategies for optimal responses.
5.   verify_and_refine_answer(answer, llm)

    LLMs sometimes generate incorrect or logically inconsistent information (hallucinations), this function takes the answer generated by the LLM and uses another LLM to:

    *   Check the answer for factual accuracy.
    *   Check for logical consistency.
    *   Refine the answer if necessary.

6.   generate_step_back(query, llm)

    This function uses an LLM to break down a complex, camera-based question into a series of logical steps. By generating these intermediate steps, the RAG system can retrieve more relevant information at each step, leading to a more comprehensive and precise answer.

In [ ]:
model_aliases = {
    "em1 mark ii": "E-M1Mk2", "e-m1 mark ii": "E-M1Mk2",
    "em5 mark ii": "E-M5Mk2", "e-m5 mark ii": "E-M5Mk2",
    "om1": "OM-1", "om-1": "OM-1",
    "em10 mark ii": "E-M10Mk2", "e-m10 mark ii": "E-M10Mk2",
}

def normalize_query(query):
    query = query.lower()
    for alias, std in model_aliases.items():
        query = query.replace(alias, std.lower())
    return query

def extract_models(query):
    q = query.lower()
    return list({std for a, std in model_aliases.items() if a in q or std.lower() in q})

def is_comparison_query(query: str) -> bool:
    models = extract_models(query)
    return len(models) >= 2

def is_general_concept_query(query: str) -> bool:
    general_keywords = ["mode", "modes", "aperture", "shutter", "manual", "auto", "focus", "exposure", "white balance", "iso", "image quality", "p mode", "a mode", "s mode", "m mode"]
    q = query.lower()
    return any(kw in q for kw in general_keywords)


def rewrite_query(user_query, llm):
    prompt = f"""Rewrite the following user query in the language typically used in a professional camera manual:
User query: "{user_query}"
Manual-style query:"""
    return llm.generate_content(prompt).text.strip()

def generate_step_back(query, llm):
    prompt = f"""Break the following camera-related question into logical steps before answering:
Question: {query}
Steps:"""
    return llm.generate_content(prompt).text.strip()

def verify_and_refine_answer(answer, llm):
    prompt = f"""Review the following answer for factual accuracy and logical consistency.

Answer:
{answer}

If the answer is fully correct and nothing is missing, respond only with:
[UNCHANGED]

Otherwise, return a corrected version of the answer."""

    review = llm.generate_content(prompt).text.strip()

    # Return original answer if review says it's fine
    if review.strip().upper() == "[UNCHANGED]":
        return answer , False
    else:
        return review , True




## 2.2. Setup Hybrid Retriever

This project uses a hybrid retriever combining BM25 and a vector retriever (built on the vectorstore).

BM25, a traditional retrieval method, which ranks documents based on term frequency and Inverse Document Frequency (TF-IDF) to evaluate the importance of the term in the document, which is useful for matching exact technical terms common in camera manuals (e.g., "ISO", "shutter").

The reason that needed vector retriever is when the input question doesn't contain the same terminology found in the manuals, vector retriever can embed queries and documents into a shared semantic space, the vector retriever can identify conceptually similar passages, even when the lexical overlap is low. Moreover, camera instructions are often interrelated. Vector search can help retrieve passages that provide broader context, rather than just sentences containing specific keywords.

Since the majority of users seek helpful comprehension instead of keyword matches, vector search is skewed more strongly (0.7), and BM25 (0.3) adds specificity for key words.

In [ ]:
bm25 = BM25Retriever.from_documents(all_chunks)
bm25.k = 4
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.3, 0.7]
)

## 2.3 Hypothetical Document Embeddings (HyDE) + Rerank Functions


Two other functions are defined to improve this RAG project.

1.   get_hypothetical_embedding()

    Using LLM to generate a hypothetical, manual-like answer. The embeddings of hypothetical answers are likely to return more semantically relevant documents than querying with the embeddings of the actual question. The benefit of this function includes overcoming vocabulary mismatch as well.

2.   rerank_documents()

    Applies *Cohere's rerank-english-v3.0 model* to rank passages on semantic relevance to the query. Since the majority of camera manuals contain duplicate content, reranking prioritizes the most relevant passages at the top, improving context quality and response precision and avoiding noise.


In [ ]:
def get_hypothetical_embedding(query, llm, embed_model):
    prompt = f"Generate a plausible manual-like answer to the question: {query}"
    hypo = llm.generate_content(prompt).text.strip()
    return embed_model.embed_query(hypo)

def rerank_documents(query, documents):
    passages = [doc.page_content for doc in documents]
    response = cohere_client.rerank(query=query, documents=passages, model="rerank-english-v3.0")
    return [documents[r.index] for r in response.results]

## 2.4 Main QA Pipeline (Hybrid + HyDE + Rerank)


The whole RAG pipeline combines the above-mentioned components into an end-to-end, multi-step process.

It begins with rewriting and normalizing the user's question, then categorizing it into one of three categories: model-specific, general concept, or comparison. This categorization determines the prompt strategy used in the answer generation.


 ***Additional Notes:***


*   The generate_step_back() function(Step 9) was turned off (steps = "") during execution. Because the empirical trials showed that most queries in this context were not complex enough to benefit from step-by-step reasoning. In some cases, activating it produced overly unnecessarily verbose or redundant outputs.

*   To support evaluation and analysis, the final output has three components: the generated answer, ranked list of supporting passages used to construct the response, context blocks sorted by camera model in organized format.









In [ ]:
def answer_with_enhanced_rag(query, llm, vectorstore, embed_model, hybrid_retriever, top_k=10):

    # Step 1: Normalize and classify the query
    normalized = normalize_query(query)
    rewritten_query = rewrite_query(normalized, llm)
    mentioned_models = extract_models(normalized)
    is_comparison = is_comparison_query(query)
    is_general = is_general_concept_query(query)

    # Step 2: Perform hybrid and HyDE retrieval
    hybrid_docs = hybrid_retriever.get_relevant_documents(rewritten_query)
    hyde_vector = get_hypothetical_embedding(rewritten_query, llm, embed_model)
    hyde_docs = vectorstore.similarity_search_by_vector(hyde_vector, k=top_k)

    # Step 3: Merge and deduplicate retrieved documents
    doc_set = {(d.metadata.get("model"), d.metadata.get("page"), d.page_content[:50]): d
               for d in hybrid_docs + hyde_docs}
    merged_docs = list(doc_set.values())

    # Step 4: Re-rank retrieved documents using Cohere Reranker
    ranked_docs = rerank_documents(rewritten_query, merged_docs)

    # Step 5: Filter documents by mentioned models (if specified in the query)
    if mentioned_models:
        ranked_docs = [doc for doc in ranked_docs if doc.metadata.get("model") in mentioned_models]

    # Step 6: Fallback if no documents were found
    if not ranked_docs:
        print("No documents found via hybrid + rerank. Using fallback keyword match...")
        ranked_docs = [doc for doc in all_chunks
                       if any(word in doc.page_content.lower() for word in normalized.split())][:5]

    # Step 7: Organize retrieved content into context blocks by model
    context_blocks = {}
    for doc in ranked_docs:
        model = doc.metadata.get("model", "Unknown")
        page = doc.metadata.get("page", "N/A")
        tag = f"[MODEL: {model}] [PAGE: {page}]"
        paragraph = f"{tag}\n{doc.page_content}"
        context_blocks.setdefault(model, []).append(paragraph)

    # Step 8: Compose final context string (limit 7 paragraphs per model)
    context = "".join(
        f"\n### {model} Manual:\n" + "\n".join(paragraphs[:7])
        for model, paragraphs in context_blocks.items()
    )

    # Step 9: Generate step-by-step reasoning plan
    steps = ""
    step_section = f"\nSteps to consider:\n{steps}" if steps else ""

    # Step 10: Select prompt template based on query type
    if is_comparison:
        prompt = f"""
You are a camera manual assistant. Compare the behavior of the following camera models
regarding the question below. Be specific about differences or similarities in their functions, menu locations, or limitations.

Context:
{context}

{step_section}

Question:
{query}

For each model, clearly state how it handles the function or feature mentioned.
Then summarize the differences.
"""
    elif is_general:
        prompt = f"""
You are a camera manual expert. Summarize and explain the following concept clearly,
based on excerpts from multiple Olympus camera manuals.

Context:
{context}

{step_section}

Question:
{query}

Give a comprehensive explanation based on all models, and include example situations.
"""
    else:
        prompt = f"""
You are a camera manual assistant. Use the following manual content to answer the user's question about a specific camera.

Context:
{context}

{step_section}

Question:
{query}

Give a clear, accurate answer based only on the context.
If the answer is not found, say "I don't know based on the manual."
"""
    # Step 11: Generate initial answer using the selected prompt
    initial = llm.generate_content(prompt).text.strip()

    # Step 12: Refine and verify the answer for factual accuracy
    final, is_changed = verify_and_refine_answer(initial, llm)

    # Step 13: Return final answer, document references, and context blocks
    return final, ranked_docs, context_blocks


# 3. Generation

A set of 13 test questions was established. Each camera model has 2–3 questions phrased using different variations of the model’s  name. The second-to-last question appears across multiple manuals, representing a cross-model issue. The final question is a general query.

In [ ]:
test_questions = [
    "How do I enable silent shooting on the OM-1?",
    "What can I do for using Super Control Panel the OM-1?",
    "How do I turn on face priority AF on the om-1?",
    "How do I transferring images to a smartphone on the E-M1 Mark II?",
    "What types of cards can be used with e-m1 mark ii?",
    "List all the lens that can be used with the e-m1 mark ii?",
    "Where do I find the ISO sensitivity settings in the E-M5 Mark II?",
    "What should I be cautions if I want to edit photos with the E-M5 Mark II?",
    "How can I format the memory card on the E-M10 Mark II?",
    "How to connect Wi-Fi on the E-M10 Mark II?",
    "Where and how can I set the self timer on the e-M10 Mark II",
    "When shooting in A mode, what does it mean if the shutter speed display is blinking? How can this be resolved?",
    "What is “P mode”? How does it differ from A/S/M modes, and in what situations is it suitable to use?"
]


*Gemini-2.0-flash* is used as it is free to accessible in the UK. It is used in multiple stages, including *rewrite_query*, *get_hypothetical_embedding*, *verify_and_refine_answer* and the *final answer generation*. Although API is free, it is subject to rate limits. Hence, a delay (*time.sleep*) is introduced between queries.

The final outputs (query, generated answer, full context) were stored in a structured format for further evaluation.

In [ ]:
import time
from google.generativeai import GenerativeModel

# Initialize Gemini model
llm = GenerativeModel("gemini-2.0-flash")

records = []

for i, question in enumerate(test_questions, 1):
    print(f"\n===================================================================================================================")
    print(f"Test Question {i}: {question}")
    print(f"===================================================================================================================")

    try:
        # Generate answer using enhanced RAG pipeline
        answer, docs, context_blocks = answer_with_enhanced_rag(
            query=question,
            llm=llm,
            vectorstore=vectorstore,
            embed_model=embedding_model,
            hybrid_retriever=hybrid_retriever,
            top_k=10
        )

        # Print answer
        print("\n>>> Answer:\n", answer)

        # Print context preview
        print("\n>>> Gemini Context Preview ===\n")
        for model, paragraphs in context_blocks.items():
            print(f"\n--- {model} Manual ---")
            for j, para in enumerate(paragraphs[:3]):
                print(f"Paragraph {j+1}: {para[:300]}...")

        # Save for RAGAs later
        all_contexts = []
        for paras in context_blocks.values():
            all_contexts.extend(paras)

        records.append({
            "question": question,
            "contexts": all_contexts,
            "answer": answer})

    except Exception as e:
        print(f"Error on question {i}: {e}")

    time.sleep(7)  # to respect rate limits


Test Question 1: How do I enable silent shooting on the OM-1?

>>> Answer:
 To enable silent shooting on the OM-1, refer to the "Shooting without shutter sound (Silent[¥] Settings)" section on page 132. You can also configure settings related to silent shooting such as "Flash Mode" (P. 127).

>>> Gemini Context Preview ===


--- OM-1 Manual ---
Paragraph 1: [MODEL: OM-1] [PAGE: 127]
[MODEL: OM-1] [PAGE: 127]
® To use the flash in silent shooting modes (P. 132), select [Allow] for [Flash Mode] in [Silent[¥] Settings] (P. 132)....
Paragraph 2: [MODEL: OM-1] [PAGE: 47]
[MODEL: OM-1] [PAGE: 47]
Shutter speeds as fast as 1/32000 s are available in [¥] (silent) mode. ES “Shooting without shutter sound (Silent[¥] Settings)” (P. 132), “Performing the sequential/self-timer shooting” (P. 126)...
Paragraph 3: [MODEL: OM-1] [PAGE: 49]
[MODEL: OM-1] [PAGE: 49]
Shutter speeds as fast as 1/32000 s are available in silent mode. ES “Shooting without shutter sound (Silent[¥] Settings)” (P. 132)...

Tes

In [ ]:
for i, record in enumerate(records):
    print(f"Record {i+1}:")
    print(f"  Question: {record.get('question')}")
    print("  Answer:")
    print(f"    {record.get('answer')}")
    print("  Contexts:")
    for j, context in enumerate(record.get('contexts', [])):
        # For brevity, printing the first 200 characters of each context, adjust as needed
        print(f"    [{j+1}] {context[:200]}...")
    print("\n")

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Path to save the records (adjust as needed)
file_path = '/content/drive/My Drive/rag_project/rag_records.pkl'

# Save the records
with open(file_path, 'wb') as f:
    pickle.dump(records, f)
print(f"Records saved to {file_path}")

In [ ]:
# Load the records
from google.colab import drive
import os
import pickle

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Path to save the records (adjust as needed)
file_path = '/content/drive/My Drive/rag_project/rag_records.pkl'
with open(file_path, 'rb') as f:
    loaded_records = pickle.load(f)

# Assign the loaded data to the 'records' variable
records = loaded_records

# 4. Evaluation

***RAGAS（Retrieval-Augmented Generation Assessment Suite）*** is used to assess the quality of generated answers.

RAGAS uses LLMs to evaluate key aspects of RAG outputs. While it defaults to OpenAI models, other LLMs can be integrated. In this project,  ***gpt-4o-mini*** was selected because it is fully compatible with the RAGAS framework.



In [ ]:
!pip install langchain_openai

RAGAS offers six metrics. This project focus on three metrics, including Faithfulness, Response Relevancy and Context Precision as these metrics are reference-free, meaning they don't require manually annotated ground-truth answers.

*   [Faithfulness:](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/faithfulness/)

  Measures consistency between the response and the retrieved context. It aims to make sure the information in the **response can be found in the retrieved context**, not a hallucination. In other words, this index is to **ensure that the generator's output is fact-based**.

*   [Response Relevancy:](https://https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/answer_relevance/)

  Measures how relevant a response is to the user input rather than assessing the accuracy of the facts. In other words, this metric ensures that the **generator's output is useful to users**.


*   [Context Precision:](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_precision/)

  Assess the ability of a retrieval system to accurately extract relevant information from a large amount of data in response to user queries. It focuses on the proportion of useful information in the retrieval results and the ability to exclude irrelevant information. In other words, it ensures that the **input provided to the generator by the retriever is of high quality.**

All three metrics are scored between 0 and 1, with higher values indicating better RAG system performance.

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
)
from ragas.llms import LangchainLLMWrapper
#from langchain_google_genai import ChatGoogleGenerativeAI\
import getpass
from langchain_openai import ChatOpenAI
from ragas.dataset_schema import EvaluationDataset

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key:")

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))

metrics = [
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm),
    LLMContextPrecisionWithoutReference(llm=evaluator_llm),
]

eval_data = [
    {
        "user_input": record['question'],
        'retrieved_contexts': record['contexts'],
        'response': record['answer']
    }
    for record in records
]

dataset = EvaluationDataset.from_list(eval_data)
results = evaluate(dataset=dataset, metrics=metrics)

print("\n=== RAGAS Evaluation Results ===")
print(results)

Enter your OpenAI API key:··········


Evaluating:   0%|          | 0/39 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[1]: KeyError('choices')



=== RAGAS Evaluation Results ===
{'faithfulness': 0.7614, 'answer_relevancy': 0.8499, 'llm_context_precision_without_reference': 0.8510}


The overall results of the RAGAS evaluation of this RAG's performance are as follows.


*   Context Precision was 0.8510, indicating excellent retrieval of relevant information, providing a **strong foundation** for response generation.
*   Answer Relevancy was 0.8499, which confirms that returned responses are extremely relevant to user queries — likely due to the advantages of the rewrite_query() function.


*   Faithfulness was slightly lesser but still upper-middle, showing that most of the responses are factually correct according to the manual. However, in certain cases, support in the retrieved context may be weak to some extent.

Let's take a closer look at the RAGAS metric scores for some questions (numbered according to the table below):


*   Record 0's Answer Relevancy is NAN, possibly because the generated answer directly points to a page rather than directly explaining how to enable it.


*   Record 2's Answer Relevancy is low because the question asked "how to," but the response provided an "introduction to Face Priority AF."

*   For Record 4, despite the answer being correct, the Context Precision score is relatively low. This might be because the first piece of information from page 156 has less direct relevance to the question.


*   Record 7's Faithfulness and Context Precision are both relatively low. This could be because the content related to "edit photo" in that question was mentioned in different forms across various pages, making it difficult to judge accurately in the end, and other important notes were not covered in the answer.



*   Record 8's Faithfulness is relatively low because the latter part of the answer is correct and consistent with the manual's content, but the former part was not directly mentioned in the manual, thus leading to a low score. The same situation also occurred in Record 10.






In [ ]:
print("\n===================== RAGAS Evaluation Results ==================================")
print(results)
print("\n\n===================== Detail of RAGAS Evaluation Results ========================")
results.to_pandas()


===================== RAGAS Evaluation Results ==================================
{'faithfulness': 0.7614, 'answer_relevancy': 0.8499, 'llm_context_precision_without_reference': 0.8510}


===================== Detail of RAGAS Evaluation Results ========================


,user_input,retrieved_contexts,response,faithfulness,answer_relevancy,llm_context_precision_without_reference
0,How do I enable silent shooting on the OM-1?,[[MODEL: OM-1] [PAGE: 127]\n[MODEL: OM-1] [PAG...,"To enable silent shooting on the OM-1, refer t...",1.000000,NaN,1.000000
1,What can I do for using Super Control Panel th...,[[MODEL: OM-1] [PAGE: 311]\n[MODEL: OM-1] [PAG...,The super control panel/LV super control panel...,0.800000,0.854350,0.676342
2,How do I turn on face priority AF on the om-1?,[[MODEL: OM-1] [PAGE: 89]\n[MODEL: OM-1] [PAGE...,The answer is mostly correct but needs some cl...,0.894737,0.000000,1.000000
3,How do I transferring images to a smartphone o...,[[MODEL: E-M1Mk2] [PAGE: 136]\n[MODEL: E-M1Mk2...,To transfer images to a smartphone on the E-M1...,1.000000,0.993229,0.931796
4,What types of cards can be used with e-m1 mark...,[[MODEL: E-M1Mk2] [PAGE: 156]\n[MODEL: E-M1Mk2...,The following types of SD memory card (commerc...,1.000000,0.872452,0.416667
5,List all the lens that can be used with the e-...,[[MODEL: E-M1Mk2] [PAGE: 149]\n[MODEL: E-M1Mk2...,Here's a breakdown of lenses compatible with t...,0.800000,0.959125,1.000000
6,Where do I find the ISO sensitivity settings i...,[[MODEL: E-M5Mk2] [PAGE: 74]\n[MODEL: E-M5Mk2]...,The answer is generally good and provides help...,0.745763,0.955468,1.000000
7,What should I be cautions if I want to edit ph...,[[MODEL: E-M5Mk2] [PAGE: 4]\n[MODEL: E-M5Mk2] ...,Here's what the manual suggests you should be ...,0.333333,0.985153,0.338828
8,How can I format the memory card on the E-M10 ...,[[MODEL: E-M10Mk2] [PAGE: 140]\n• Select [Clea...,The answer is missing crucial steps. Here's a ...,0.333333,0.981503,1.000000
9,How to connect Wi-Fi on the E-M10 Mark II?,[[MODEL: E-M10Mk2] [PAGE: 120]\n[MODEL: E-M10M...,1. Select [Wi-Fi Connect Settings] and press I...,1.000000,0.856182,1.000000


# 5. Limitation for this RAGs and Future Improvements

## 5.1 Limitations of the Current RAG System


*   Challenges in Processing Symbolic and Visual Semantics:

  One of the main limitations of this RAG system lies in its insufficient ability to comprehend non-textual elements within user manuals. For instance, in one of our test questions, the phrase 'Silent[¥] Settings' appeared. The '¥' symbol in this context should represent a 'heart' icon, intended to guide users to the corresponding setting in the camera menu. However, the current RAG system can only process textual information and is unable to understand and correlate the semantic meaning conveyed by these visual symbols. This limitation prevents it from accurately guiding users through operations.

*   Issues with Document Redundancy and Context Confusion

  Cameras feature various operating modes (e.g., P, M, A, S modes), and many functionalities are mentioned across different modes or sections. Additionally, for user convenience, certain functions are repeatedly presented in various ways or locations throughout the manual. This content redundancy can lead to the current RAG system retrieving irrelevant or incorrect context chunks during the retrieval phase.
  
Although various methods (such as OCR and re-ranking) have been employed in this project to mitigate the aforementioned limitations, optimal results have yet to be achieved.

## 5.2 Future Improvements: Advanced Multi-modal RAG

  In the early stages of this project, using [multimodal RAG](https://huggingface.co/learn/cookbook/multimodal_rag_using_document_retrieval_and_vlms), which would be effective for PDFs containing images, was considered but was unable to continue due to insufficient computing resources.
  
  If more computing resources can be invested in the future, more advanced multimodal RAG models can be researched and implemented. Through this approach, the RAG system will be able to analyse the illustrations in the manual, identify the symbols as the correct camera icons, and establish associations, thereby providing more accurate and easy-to-understand operating instructions.


